In [1]:
import pandas as pd
import numpy as np
import itertools
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import statsmodels.api as sm
from econml.dml import CausalForestDML
from sklearn.ensemble import GradientBoostingRegressor,RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV, RidgeCV, ElasticNetCV
from sklearn.utils.class_weight import compute_sample_weight
from lightgbm import LGBMRegressor
from econml.dml import LinearDML

g:\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 数据预处理
df = pd.read_csv(r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\all_data_final_with_IMR_rename.csv')

required_cols = ['ExcellenceRate','PassRate', 'r1h','rfh', 'r3h', 'FluctMonth', 'FluctIntra', 'FluctTenDays',
                 'SchType', 'SchLocation', 'Gender', 'IMR','Grade', 'District', 'SchYear','Date']
df = df.dropna(subset=required_cols)

# 核心自变量归一化
scaler = StandardScaler()
for var in ['r1h', 'r3h', 'FluctMonth']:
    df[f'Norm{var}'] = scaler.fit_transform(df[[var]])


#缩尾处理进行鲁棒性check
# 执行缩尾处理 (Winsorization)
# 理由：这能防止极端降水值过度左右回归系数，同时保留降雨分布的主要特征。
core_vars = ['r1h', 'r3h', 'FluctMonth']
for var in core_vars:
    lower_limit = df[var].quantile(0.01)#0.01 0.05
    upper_limit = df[var].quantile(0.99)#0.99 0.95
    lower_limit2 = df[var].quantile(0.05)#0.01 0.05
    upper_limit2 = df[var].quantile(0.95)#0.99 0.95

    df[f'NormWins{var}'] = df[var].clip(lower_limit, upper_limit)
    df[f'NormWins{var}2'] = df[var].clip(lower_limit2, upper_limit2)
for var in core_vars:
    df[f'NormWins{var}'] = scaler.fit_transform(df[[f'NormWins{var}']])
    df[f'NormWins{var}2'] = scaler.fit_transform(df[[f'NormWins{var}2']])



df_ml = df.copy()
# 固定效应转哑变量 (W_cols)
df_ml = pd.get_dummies(df_ml, columns=['Grade','District','SchYear','Date'], drop_first=True)

# 因变量为PassRate

In [3]:
# 变量分类定义
Y_col = 'PassRate'
T_cols = ['Normr1h', 'Normr3h', 'NormFluctMonth']
X_cols =  ['SchType', 'SchLocation', 'Gender'] # 调节变量 ['SchType', 'SchLocation', 'Gender']  ['SchType', 'SchLocation']  # 这里修改后下面对应的def describe_case 也要修改
#W_cols = ['IMR'] + [c for c in df_ml.columns if any(p in c for p in ['Grade_', 'District_', 'SchYear_'])]

# 循环分析每个处理变量
# 用于存储汇总结果的列表
final_summary_tables = []

for t_var in T_cols:
    print(f"\n{'='*20} 正在进行 LinearDML 分析: {t_var} {'='*20}")
    
    if t_var=='Normr1h':   #$$$$
        W_cols=['IMR','Normr3h','NormFluctMonth'] + [c for c in df_ml.columns if any(p in c for p in ['Grade_', 'District_', 'SchYear_'])] #, 'District_', 'SchYear_'      , 'District_', 'Date_'
    elif t_var=='Normr3h':
        W_cols=['IMR','Normr1h','NormFluctMonth'] + [c for c in df_ml.columns if any(p in c for p in ['Grade_', 'District_', 'SchYear_'])] #, 'District_', 'SchYear_'      , 'District_', 'Date_'
    else:
        W_cols=['IMR','Normr1h', 'Normr3h'] + [c for c in df_ml.columns if any(p in c for p in ['Grade_', 'District_', 'SchYear_'])] #, 'District_', 'SchYear_'    , 'District_', 'Date_'


    # A. 构建并拟合 LinearDML
    seed = 120      #$$$$
    est = LinearDML(
        model_y=LassoCV(random_state=seed),##@@@
        model_t=LassoCV(random_state=seed),
        discrete_treatment=False,
        cv=3,
        random_state=seed
    )
    

    # # 创建一个组合标签，例如 "1-0-1" 代表 "私立-农村-男生"
    # combined_str = df_ml[X_cols].astype(str).agg('-'.join, axis=1)
    # # 自动计算平衡权重：样本越少的组合，权重越高
    # # compute_sample_weight 会确保所有类别的权重总和相等
    # sw = compute_sample_weight(class_weight='balanced', y=combined_str)

    # 拟合
    # 首先将Y和T对W与X回归，得到残差；然后T的残差对Y的残差回归，系数即为调节效应tao(x)；然后再用X各个特征回归这个tao(x)；
    # 得到的X的各个特征的系数即为其各个特征对“T影响Y”的贡献，亦即T:X交互项；
    
    # weights = np.where(df_ml['SchLocation'] == 1, 1/0.35, 1/0.65)
    est.fit(df_ml[Y_col], df_ml[t_var], X=df_ml[X_cols], W=df_ml[W_cols]) #, sample_weight=sw
    

    # B. 计算个体处理效应 (HTE)
    hte_values = est.effect(df_ml[X_cols])
    df_ml[f'HTE_{t_var}'] = hte_values
    
    # C. 全模型统计摘要 (这是 LinearDML 的核心优势，直接给出系数 P 值)
    print("\n--- 1. LinearDML 阶段二系数汇总 (直接对应交互项系数) ---")
    print(est.summary()) # 这一步直接产出类似回归表的统计结果

    # D. 显著效应样本占比
    inf_full = est.effect_inference(df_ml[X_cols])
    df_ml[f'p_value_{t_var}'] = inf_full.pvalue()
    print(f"\n--- 2. 个体层面显著效应样本占比 (p < 0.05): {(df_ml[f'p_value_{t_var}'] < 0.05).mean():.2%}")

    # E. 各变量组合的具体效应预测
    print("\n--- 3. 各变量组合的具体效应预测 ---")
    combinations = list(itertools.product([0, 1], repeat=len(X_cols)))
    df_comb = pd.DataFrame(combinations, columns=X_cols)
    inf_comb = est.effect_inference(df_comb)
    df_comb['treatment_effect'] = est.effect(df_comb)
    df_comb['p_value'] = inf_comb.pvalue()
    
    def describe_case(row):
        st = "私立" if row['SchType'] == 1 else "公立"
        sl = "城市" if row['SchLocation'] == 1 else "农村"
        ge = "男生" if row['Gender'] == 1 else "女生"
        return f"{st}-{sl}-{ge}" #f"{st}-{sl}" #f"{st}-{sl}-{ge}"
    
    
    df_comb['Group_Desc'] = df_comb.apply(describe_case, axis=1)
    print(df_comb[['Group_Desc', 'treatment_effect', 'p_value']])

    # F. 分组汇总表生成
    sub_tables = []
    for x_col in X_cols:
        grouped = df_ml.groupby(x_col)[f'HTE_{t_var}'].agg(['mean', 'std', 'count']).reset_index()
        grouped.columns = ['Category_Value', 'Mean_Effect', 'Std_Dev', 'Sample_Size']
        grouped['Moderator'], grouped['Treatment'] = x_col, t_var
        sub_tables.append(grouped)
    final_summary_tables.append(pd.concat(sub_tables, ignore_index=True))


# 结果保存
final_report = pd.concat(final_summary_tables, ignore_index=True)
output_path = r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\Causal_Forest_Final_Results_PassRate.csv'
final_report.to_csv(output_path, index=False)
print(f"\n全部分析完成，导出路径: {output_path}")


==================== 正在进行 LinearDML 分析: Normr1h ====================

--- 1. LinearDML 阶段二系数汇总 (直接对应交互项系数) ---
                      Coefficient Results                       
            point_estimate stderr zstat pvalue ci_lower ci_upper
----------------------------------------------------------------
SchType              0.001  0.005 0.228   0.82    -0.01    0.012
SchLocation          0.002  0.003 0.539   0.59   -0.005    0.009
Gender                 0.0  0.003 0.042  0.967   -0.006    0.006
                       CATE Intercept Results                      
               point_estimate stderr zstat pvalue ci_lower ci_upper
-------------------------------------------------------------------
cate_intercept          0.001  0.003   0.5  0.617   -0.004    0.006
-------------------------------------------------------------------

<sub>A linear parametric conditional average treatment effect (CATE) model was fitted:
$Y = \Theta(X)\cdot T + g(X, W) + \epsilon$
where for every outcome $i

# 因变量为PassRate 缩尾处理稳健性检验

In [4]:
# 变量分类定义
Y_col = 'PassRate'
T_cols = ['NormWinsr1h', 'NormWinsr3h', 'NormWinsFluctMonth']
X_cols =  ['SchType', 'SchLocation', 'Gender'] # 调节变量 ['SchType', 'SchLocation', 'Gender']  ['SchType', 'SchLocation']  # 这里修改后下面对应的def describe_case 也要修改
#W_cols = ['IMR'] + [c for c in df_ml.columns if any(p in c for p in ['Grade_', 'District_', 'SchYear_'])]

# 循环分析每个处理变量
# 用于存储汇总结果的列表
final_summary_tables = []

for t_var in T_cols:
    print(f"\n{'='*20} 正在进行 LinearDML 分析: {t_var} {'='*20}")
    
    if t_var=='NormWinsr1h':   #$$$$
        W_cols=['IMR','NormWinsr3h','NormWinsFluctMonth'] + [c for c in df_ml.columns if any(p in c for p in ['Grade_', 'District_', 'SchYear_'])] #, 'District_', 'SchYear_'      , 'District_', 'Date_'
    elif t_var=='NormWinsr3h':
        W_cols=['IMR','NormWinsr1h','NormWinsFluctMonth'] + [c for c in df_ml.columns if any(p in c for p in ['Grade_', 'District_', 'SchYear_'])] #, 'District_', 'SchYear_'      , 'District_', 'Date_'
    else:
        W_cols=['IMR','NormWinsr1h', 'NormWinsr3h'] + [c for c in df_ml.columns if any(p in c for p in ['Grade_', 'District_', 'SchYear_'])] #, 'District_', 'SchYear_'    , 'District_', 'Date_'


    # A. 构建并拟合 LinearDML
    seed = 120      #$$$$
    est = LinearDML(
        model_y=LassoCV(random_state=seed),##@@@
        model_t=LassoCV(random_state=seed),
        # model_y=RandomForestRegressor(n_estimators=100, random_state=seed),
        # model_t=RandomForestRegressor(n_estimators=100, random_state=seed),
        # model_y=GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=seed),
        # model_t=GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=seed),
        # model_y=RidgeCV(), 
        # model_t=RidgeCV(),
        discrete_treatment=False,
        cv=3,
        random_state=seed
    )
    

    # # 创建一个组合标签，例如 "1-0-1" 代表 "私立-农村-男生"
    # combined_str = df_ml[X_cols].astype(str).agg('-'.join, axis=1)
    # # 自动计算平衡权重：样本越少的组合，权重越高
    # # compute_sample_weight 会确保所有类别的权重总和相等
    # sw = compute_sample_weight(class_weight='balanced', y=combined_str)

    # 拟合
    # 首先将Y和T对W与X回归，得到残差；然后T的残差对Y的残差回归，系数即为调节效应tao(x)；然后再用X各个特征回归这个tao(x)；
    # 得到的X的各个特征的系数即为其各个特征对“T影响Y”的贡献，亦即T:X交互项；
    
    # weights = np.where(df_ml['SchLocation'] == 1, 1/0.35, 1/0.65)
    est.fit(df_ml[Y_col], df_ml[t_var], X=df_ml[X_cols], W=df_ml[W_cols]) #, sample_weight=sw
    

    # B. 计算个体处理效应 (HTE)
    hte_values = est.effect(df_ml[X_cols])
    df_ml[f'HTE_{t_var}'] = hte_values
    
    # C. 全模型统计摘要 (这是 LinearDML 的核心优势，直接给出系数 P 值)
    print("\n--- 1. LinearDML 阶段二系数汇总 (直接对应交互项系数) ---")
    print(est.summary()) # 这一步直接产出类似回归表的统计结果

    # D. 显著效应样本占比
    inf_full = est.effect_inference(df_ml[X_cols])
    df_ml[f'p_value_{t_var}'] = inf_full.pvalue()
    print(f"\n--- 2. 个体层面显著效应样本占比 (p < 0.05): {(df_ml[f'p_value_{t_var}'] < 0.05).mean():.2%}")

    # E. 各变量组合的具体效应预测
    print("\n--- 3. 各变量组合的具体效应预测 ---")
    combinations = list(itertools.product([0, 1], repeat=len(X_cols)))
    df_comb = pd.DataFrame(combinations, columns=X_cols)
    inf_comb = est.effect_inference(df_comb)
    df_comb['treatment_effect'] = est.effect(df_comb)
    df_comb['p_value'] = inf_comb.pvalue()
    
    def describe_case(row):
        st = "私立" if row['SchType'] == 1 else "公立"
        sl = "城市" if row['SchLocation'] == 1 else "农村"
        ge = "男生" if row['Gender'] == 1 else "女生"
        return f"{st}-{sl}-{ge}" #f"{st}-{sl}" #f"{st}-{sl}-{ge}"
    
    
    df_comb['Group_Desc'] = df_comb.apply(describe_case, axis=1)
    print(df_comb[['Group_Desc', 'treatment_effect', 'p_value']])

    # F. 分组汇总表生成
    sub_tables = []
    for x_col in X_cols:
        grouped = df_ml.groupby(x_col)[f'HTE_{t_var}'].agg(['mean', 'std', 'count']).reset_index()
        grouped.columns = ['Category_Value', 'Mean_Effect', 'Std_Dev', 'Sample_Size']
        grouped['Moderator'], grouped['Treatment'] = x_col, t_var
        sub_tables.append(grouped)
    final_summary_tables.append(pd.concat(sub_tables, ignore_index=True))


# 结果保存
final_report = pd.concat(final_summary_tables, ignore_index=True)
output_path = r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\Causal_Forest_Final_Results_PassRate.csv'
final_report.to_csv(output_path, index=False)
print(f"\n全部分析完成，导出路径: {output_path}")


==================== 正在进行 LinearDML 分析: NormWinsr1h ====================

--- 1. LinearDML 阶段二系数汇总 (直接对应交互项系数) ---
                      Coefficient Results                       
            point_estimate stderr zstat pvalue ci_lower ci_upper
----------------------------------------------------------------
SchType              0.001  0.005 0.204  0.838    -0.01    0.012
SchLocation          0.002  0.003 0.557  0.578   -0.005    0.009
Gender                 0.0  0.003 0.053  0.958   -0.006    0.006
                       CATE Intercept Results                      
               point_estimate stderr zstat pvalue ci_lower ci_upper
-------------------------------------------------------------------
cate_intercept          0.001  0.003 0.508  0.611   -0.004    0.006
-------------------------------------------------------------------

<sub>A linear parametric conditional average treatment effect (CATE) model was fitted:
$Y = \Theta(X)\cdot T + g(X, W) + \epsilon$
where for every outcom

# 因变量为PassRate 缩尾处理稳健性检验2

In [5]:
# 变量分类定义
Y_col = 'PassRate'
T_cols = ['NormWinsr1h2', 'NormWinsr3h2', 'NormWinsFluctMonth2']
X_cols =  ['SchType', 'SchLocation', 'Gender'] # 调节变量 ['SchType', 'SchLocation', 'Gender']  ['SchType', 'SchLocation']  # 这里修改后下面对应的def describe_case 也要修改
#W_cols = ['IMR'] + [c for c in df_ml.columns if any(p in c for p in ['Grade_', 'District_', 'SchYear_'])]

# 循环分析每个处理变量
# 用于存储汇总结果的列表
final_summary_tables = []

for t_var in T_cols:
    print(f"\n{'='*20} 正在进行 LinearDML 分析: {t_var} {'='*20}")
    
    if t_var=='NormWinsr1h2':   #$$$$
        W_cols=['IMR','NormWinsr3h2','NormWinsFluctMonth2'] + [c for c in df_ml.columns if any(p in c for p in ['Grade_', 'District_', 'SchYear_'])] #, 'District_', 'SchYear_'      , 'District_', 'Date_'
    elif t_var=='NormWinsr3h2':
        W_cols=['IMR','NormWinsr1h2','NormWinsFluctMonth2'] + [c for c in df_ml.columns if any(p in c for p in ['Grade_', 'District_', 'SchYear_'])] #, 'District_', 'SchYear_'      , 'District_', 'Date_'
    else:
        W_cols=['IMR','NormWinsr1h2', 'NormWinsr3h2'] + [c for c in df_ml.columns if any(p in c for p in ['Grade_', 'District_', 'SchYear_'])] #, 'District_', 'SchYear_'    , 'District_', 'Date_'


    # A. 构建并拟合 LinearDML
    seed = 120      #$$$$
    est = LinearDML(
        model_y=LassoCV(random_state=seed),##@@@
        model_t=LassoCV(random_state=seed),
        # model_y=RandomForestRegressor(n_estimators=100, random_state=seed),
        # model_t=RandomForestRegressor(n_estimators=100, random_state=seed),
        # model_y=GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=seed),
        # model_t=GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=seed),
        # model_y=RidgeCV(), 
        # model_t=RidgeCV(),
        discrete_treatment=False,
        cv=3,
        random_state=seed
    )
    

    # # 创建一个组合标签，例如 "1-0-1" 代表 "私立-农村-男生"
    # combined_str = df_ml[X_cols].astype(str).agg('-'.join, axis=1)
    # # 自动计算平衡权重：样本越少的组合，权重越高
    # # compute_sample_weight 会确保所有类别的权重总和相等
    # sw = compute_sample_weight(class_weight='balanced', y=combined_str)

    # 拟合
    # 首先将Y和T对W与X回归，得到残差；然后T的残差对Y的残差回归，系数即为调节效应tao(x)；然后再用X各个特征回归这个tao(x)；
    # 得到的X的各个特征的系数即为其各个特征对“T影响Y”的贡献，亦即T:X交互项；
    
    # weights = np.where(df_ml['SchLocation'] == 1, 1/0.35, 1/0.65)
    est.fit(df_ml[Y_col], df_ml[t_var], X=df_ml[X_cols], W=df_ml[W_cols]) #, sample_weight=sw
    

    # B. 计算个体处理效应 (HTE)
    hte_values = est.effect(df_ml[X_cols])
    df_ml[f'HTE_{t_var}'] = hte_values
    
    # C. 全模型统计摘要 (这是 LinearDML 的核心优势，直接给出系数 P 值)
    print("\n--- 1. LinearDML 阶段二系数汇总 (直接对应交互项系数) ---")
    print(est.summary()) # 这一步直接产出类似回归表的统计结果

    # D. 显著效应样本占比
    inf_full = est.effect_inference(df_ml[X_cols])
    df_ml[f'p_value_{t_var}'] = inf_full.pvalue()
    print(f"\n--- 2. 个体层面显著效应样本占比 (p < 0.05): {(df_ml[f'p_value_{t_var}'] < 0.05).mean():.2%}")

    # E. 各变量组合的具体效应预测
    print("\n--- 3. 各变量组合的具体效应预测 ---")
    combinations = list(itertools.product([0, 1], repeat=len(X_cols)))
    df_comb = pd.DataFrame(combinations, columns=X_cols)
    inf_comb = est.effect_inference(df_comb)
    df_comb['treatment_effect'] = est.effect(df_comb)
    df_comb['p_value'] = inf_comb.pvalue()
    
    def describe_case(row):
        st = "私立" if row['SchType'] == 1 else "公立"
        sl = "城市" if row['SchLocation'] == 1 else "农村"
        ge = "男生" if row['Gender'] == 1 else "女生"
        return f"{st}-{sl}-{ge}" #f"{st}-{sl}" #f"{st}-{sl}-{ge}"
    
    
    df_comb['Group_Desc'] = df_comb.apply(describe_case, axis=1)
    print(df_comb[['Group_Desc', 'treatment_effect', 'p_value']])

    # F. 分组汇总表生成
    sub_tables = []
    for x_col in X_cols:
        grouped = df_ml.groupby(x_col)[f'HTE_{t_var}'].agg(['mean', 'std', 'count']).reset_index()
        grouped.columns = ['Category_Value', 'Mean_Effect', 'Std_Dev', 'Sample_Size']
        grouped['Moderator'], grouped['Treatment'] = x_col, t_var
        sub_tables.append(grouped)
    final_summary_tables.append(pd.concat(sub_tables, ignore_index=True))


# 结果保存
final_report = pd.concat(final_summary_tables, ignore_index=True)
output_path = r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\Causal_Forest_Final_Results_PassRate.csv'
final_report.to_csv(output_path, index=False)
print(f"\n全部分析完成，导出路径: {output_path}")


==================== 正在进行 LinearDML 分析: NormWinsr1h2 ====================

--- 1. LinearDML 阶段二系数汇总 (直接对应交互项系数) ---
                       Coefficient Results                       
            point_estimate stderr zstat  pvalue ci_lower ci_upper
-----------------------------------------------------------------
SchType               -0.0  0.006 -0.063   0.95   -0.011    0.011
SchLocation          0.003  0.003  0.822  0.411   -0.004     0.01
Gender                 0.0  0.003  0.154  0.878   -0.006    0.007
                       CATE Intercept Results                      
               point_estimate stderr zstat pvalue ci_lower ci_upper
-------------------------------------------------------------------
cate_intercept          0.001  0.003 0.266   0.79   -0.004    0.006
-------------------------------------------------------------------

<sub>A linear parametric conditional average treatment effect (CATE) model was fitted:
$Y = \Theta(X)\cdot T + g(X, W) + \epsilon$
where for every

# 因变量为ExcellenceRate

In [6]:
# 变量分类定义
Y_col = 'ExcellenceRate'
T_cols = ['Normr1h', 'Normr3h', 'NormFluctMonth']
X_cols =  ['SchType', 'SchLocation', 'Gender'] # 调节变量 ['SchType', 'SchLocation', 'Gender']  ['SchType', 'SchLocation']  # 这里修改后下面对应的def describe_case 也要修改
#W_cols = ['IMR'] + [c for c in df_ml.columns if any(p in c for p in ['Grade_', 'District_', 'SchYear_'])]

# 循环分析每个处理变量
# 用于存储汇总结果的列表
final_summary_tables = []

for t_var in T_cols:
    print(f"\n{'='*20} 正在进行 LinearDML 分析: {t_var} {'='*20}")
    
    if t_var=='Normr1h':   #$$$$
        W_cols=['IMR','Normr3h','NormFluctMonth'] + [c for c in df_ml.columns if any(p in c for p in ['Grade_', 'District_', 'SchYear_'])] #, 'District_', 'SchYear_'      , 'District_', 'Date_'
    elif t_var=='Normr3h':
        W_cols=['IMR','Normr1h','NormFluctMonth'] + [c for c in df_ml.columns if any(p in c for p in ['Grade_', 'District_', 'SchYear_'])] #, 'District_', 'SchYear_'      , 'District_', 'Date_'
    else:
        W_cols=['IMR','Normr1h', 'Normr3h'] + [c for c in df_ml.columns if any(p in c for p in ['Grade_', 'District_', 'SchYear_'])] #, 'District_', 'SchYear_'    , 'District_', 'Date_'


    # A. 构建并拟合 LinearDML
    seed = 120      #$$$$
    est = LinearDML(
        model_y=LassoCV(random_state=seed),
        model_t=LassoCV(random_state=seed),
        discrete_treatment=False,
        cv=3,
        random_state=seed
    )
    

    # # 创建一个组合标签，例如 "1-0-1" 代表 "私立-农村-男生"
    # combined_str = df_ml[X_cols].astype(str).agg('-'.join, axis=1)
    # # 自动计算平衡权重：样本越少的组合，权重越高
    # # compute_sample_weight 会确保所有类别的权重总和相等
    # sw = compute_sample_weight(class_weight='balanced', y=combined_str)

    # 拟合
    # 首先将Y和T对W与X回归，得到残差；然后T的残差对Y的残差回归，系数即为调节效应tao(x)；然后再用X各个特征回归这个tao(x)；
    # 得到的X的各个特征的系数即为其各个特征对“T影响Y”的贡献，亦即T:X交互项；
    
    # weights = np.where(df_ml['SchLocation'] == 1, 1/0.35, 1/0.65)
    est.fit(df_ml[Y_col], df_ml[t_var], X=df_ml[X_cols], W=df_ml[W_cols]) #, sample_weight=sw
    

    # B. 计算个体处理效应 (HTE)
    hte_values = est.effect(df_ml[X_cols])
    df_ml[f'HTE_{t_var}'] = hte_values
    
    # C. 全模型统计摘要 (这是 LinearDML 的核心优势，直接给出系数 P 值)
    print("\n--- 1. LinearDML 阶段二系数汇总 (直接对应交互项系数) ---")
    print(est.summary()) # 这一步直接产出类似回归表的统计结果

    # D. 显著效应样本占比
    inf_full = est.effect_inference(df_ml[X_cols])
    df_ml[f'p_value_{t_var}'] = inf_full.pvalue()
    print(f"\n--- 2. 个体层面显著效应样本占比 (p < 0.05): {(df_ml[f'p_value_{t_var}'] < 0.05).mean():.2%}")

    # E. 各变量组合的具体效应预测
    print("\n--- 3. 各变量组合的具体效应预测 ---")
    combinations = list(itertools.product([0, 1], repeat=len(X_cols)))
    df_comb = pd.DataFrame(combinations, columns=X_cols)
    inf_comb = est.effect_inference(df_comb)
    df_comb['treatment_effect'] = est.effect(df_comb)
    df_comb['p_value'] = inf_comb.pvalue()
    
    def describe_case(row):
        st = "私立" if row['SchType'] == 1 else "公立"
        sl = "城市" if row['SchLocation'] == 1 else "农村"
        ge = "男生" if row['Gender'] == 1 else "女生"
        return f"{st}-{sl}-{ge}" #f"{st}-{sl}" #f"{st}-{sl}-{ge}"
    
    
    df_comb['Group_Desc'] = df_comb.apply(describe_case, axis=1)
    print(df_comb[['Group_Desc', 'treatment_effect', 'p_value']])

    # F. 分组汇总表生成
    sub_tables = []
    for x_col in X_cols:
        grouped = df_ml.groupby(x_col)[f'HTE_{t_var}'].agg(['mean', 'std', 'count']).reset_index()
        grouped.columns = ['Category_Value', 'Mean_Effect', 'Std_Dev', 'Sample_Size']
        grouped['Moderator'], grouped['Treatment'] = x_col, t_var
        sub_tables.append(grouped)
    final_summary_tables.append(pd.concat(sub_tables, ignore_index=True))


# 结果保存
final_report = pd.concat(final_summary_tables, ignore_index=True)
output_path = r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\Causal_Forest_Final_Results_ExcellenceRate.csv'
final_report.to_csv(output_path, index=False)
print(f"\n全部分析完成，导出路径: {output_path}")


==================== 正在进行 LinearDML 分析: Normr1h ====================

--- 1. LinearDML 阶段二系数汇总 (直接对应交互项系数) ---
                       Coefficient Results                       
            point_estimate stderr zstat  pvalue ci_lower ci_upper
-----------------------------------------------------------------
SchType             -0.037  0.007 -5.369    0.0    -0.05   -0.023
SchLocation         -0.001  0.004 -0.169  0.866   -0.008    0.007
Gender               0.001  0.003  0.243  0.808   -0.006    0.008
                       CATE Intercept Results                      
               point_estimate stderr zstat pvalue ci_lower ci_upper
-------------------------------------------------------------------
cate_intercept          0.004  0.003 1.338  0.181   -0.002    0.009
-------------------------------------------------------------------

<sub>A linear parametric conditional average treatment effect (CATE) model was fitted:
$Y = \Theta(X)\cdot T + g(X, W) + \epsilon$
where for every outc

In [7]:
# 变量分类定义
Y_col = 'ExcellenceRate'
T_cols = ['NormWinsr1h', 'NormWinsr3h', 'NormWinsFluctMonth']
X_cols =  ['SchType', 'SchLocation', 'Gender'] # 调节变量 ['SchType', 'SchLocation', 'Gender']  ['SchType', 'SchLocation']  # 这里修改后下面对应的def describe_case 也要修改
#W_cols = ['IMR'] + [c for c in df_ml.columns if any(p in c for p in ['Grade_', 'District_', 'SchYear_'])]

# 循环分析每个处理变量
# 用于存储汇总结果的列表
final_summary_tables = []

for t_var in T_cols:
    print(f"\n{'='*20} 正在进行 LinearDML 分析: {t_var} {'='*20}")
    
    if t_var=='NormWinsr1h':   #$$$$
        W_cols=['IMR','NormWinsr3h','NormWinsFluctMonth'] + [c for c in df_ml.columns if any(p in c for p in ['Grade_', 'District_', 'SchYear_'])] #, 'District_', 'SchYear_'      , 'District_', 'Date_'
    elif t_var=='NormWinsr3h':
        W_cols=['IMR','NormWinsr1h','NormWinsFluctMonth'] + [c for c in df_ml.columns if any(p in c for p in ['Grade_', 'District_', 'SchYear_'])] #, 'District_', 'SchYear_'      , 'District_', 'Date_'
    else:
        W_cols=['IMR','NormWinsr1h', 'NormWinsr3h'] + [c for c in df_ml.columns if any(p in c for p in ['Grade_', 'District_', 'SchYear_'])] #, 'District_', 'SchYear_'    , 'District_', 'Date_'


    # A. 构建并拟合 LinearDML
    seed = 120      #$$$$
    est = LinearDML(
        model_y=LassoCV(random_state=seed),##@@@
        model_t=LassoCV(random_state=seed),
        # model_y=RandomForestRegressor(n_estimators=100, random_state=seed),
        # model_t=RandomForestRegressor(n_estimators=100, random_state=seed),
        # model_y=GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=seed),
        # model_t=GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=seed),
        # model_y=RidgeCV(), 
        # model_t=RidgeCV(),
        discrete_treatment=False,
        cv=3,
        random_state=seed
    )
    

    # # 创建一个组合标签，例如 "1-0-1" 代表 "私立-农村-男生"
    # combined_str = df_ml[X_cols].astype(str).agg('-'.join, axis=1)
    # # 自动计算平衡权重：样本越少的组合，权重越高
    # # compute_sample_weight 会确保所有类别的权重总和相等
    # sw = compute_sample_weight(class_weight='balanced', y=combined_str)

    # 拟合
    # 首先将Y和T对W与X回归，得到残差；然后T的残差对Y的残差回归，系数即为调节效应tao(x)；然后再用X各个特征回归这个tao(x)；
    # 得到的X的各个特征的系数即为其各个特征对“T影响Y”的贡献，亦即T:X交互项；
    
    # weights = np.where(df_ml['SchLocation'] == 1, 1/0.35, 1/0.65)
    est.fit(df_ml[Y_col], df_ml[t_var], X=df_ml[X_cols], W=df_ml[W_cols]) #, sample_weight=sw
    

    # B. 计算个体处理效应 (HTE)
    hte_values = est.effect(df_ml[X_cols])
    df_ml[f'HTE_{t_var}'] = hte_values
    
    # C. 全模型统计摘要 (这是 LinearDML 的核心优势，直接给出系数 P 值)
    print("\n--- 1. LinearDML 阶段二系数汇总 (直接对应交互项系数) ---")
    print(est.summary()) # 这一步直接产出类似回归表的统计结果

    # D. 显著效应样本占比
    inf_full = est.effect_inference(df_ml[X_cols])
    df_ml[f'p_value_{t_var}'] = inf_full.pvalue()
    print(f"\n--- 2. 个体层面显著效应样本占比 (p < 0.05): {(df_ml[f'p_value_{t_var}'] < 0.05).mean():.2%}")

    # E. 各变量组合的具体效应预测
    print("\n--- 3. 各变量组合的具体效应预测 ---")
    combinations = list(itertools.product([0, 1], repeat=len(X_cols)))
    df_comb = pd.DataFrame(combinations, columns=X_cols)
    inf_comb = est.effect_inference(df_comb)
    df_comb['treatment_effect'] = est.effect(df_comb)
    df_comb['p_value'] = inf_comb.pvalue()
    
    def describe_case(row):
        st = "私立" if row['SchType'] == 1 else "公立"
        sl = "城市" if row['SchLocation'] == 1 else "农村"
        ge = "男生" if row['Gender'] == 1 else "女生"
        return f"{st}-{sl}-{ge}" #f"{st}-{sl}" #f"{st}-{sl}-{ge}"
    
    
    df_comb['Group_Desc'] = df_comb.apply(describe_case, axis=1)
    print(df_comb[['Group_Desc', 'treatment_effect', 'p_value']])

    # F. 分组汇总表生成
    sub_tables = []
    for x_col in X_cols:
        grouped = df_ml.groupby(x_col)[f'HTE_{t_var}'].agg(['mean', 'std', 'count']).reset_index()
        grouped.columns = ['Category_Value', 'Mean_Effect', 'Std_Dev', 'Sample_Size']
        grouped['Moderator'], grouped['Treatment'] = x_col, t_var
        sub_tables.append(grouped)
    final_summary_tables.append(pd.concat(sub_tables, ignore_index=True))


# 结果保存
final_report = pd.concat(final_summary_tables, ignore_index=True)
output_path = r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\Causal_Forest_Final_Results_ExcellenceRate.csv'
final_report.to_csv(output_path, index=False)
print(f"\n全部分析完成，导出路径: {output_path}")


==================== 正在进行 LinearDML 分析: NormWinsr1h ====================

--- 1. LinearDML 阶段二系数汇总 (直接对应交互项系数) ---
                       Coefficient Results                       
            point_estimate stderr zstat  pvalue ci_lower ci_upper
-----------------------------------------------------------------
SchType             -0.037  0.007 -5.364    0.0    -0.05   -0.023
SchLocation         -0.001  0.004  -0.16  0.873   -0.008    0.007
Gender               0.001  0.003  0.246  0.806   -0.006    0.008
                       CATE Intercept Results                      
               point_estimate stderr zstat pvalue ci_lower ci_upper
-------------------------------------------------------------------
cate_intercept          0.004  0.003 1.406   0.16   -0.002    0.009
-------------------------------------------------------------------

<sub>A linear parametric conditional average treatment effect (CATE) model was fitted:
$Y = \Theta(X)\cdot T + g(X, W) + \epsilon$
where for every 

In [8]:
# 变量分类定义
Y_col = 'ExcellenceRate'
T_cols = ['NormWinsr1h2', 'NormWinsr3h2', 'NormWinsFluctMonth2']
X_cols =  ['SchType', 'SchLocation', 'Gender'] # 调节变量 ['SchType', 'SchLocation', 'Gender']  ['SchType', 'SchLocation']  # 这里修改后下面对应的def describe_case 也要修改
#W_cols = ['IMR'] + [c for c in df_ml.columns if any(p in c for p in ['Grade_', 'District_', 'SchYear_'])]

# 循环分析每个处理变量
# 用于存储汇总结果的列表
final_summary_tables = []

for t_var in T_cols:
    print(f"\n{'='*20} 正在进行 LinearDML 分析: {t_var} {'='*20}")
    
    if t_var=='NormWinsr1h2':   #$$$$
        W_cols=['IMR','NormWinsr3h2','NormWinsFluctMonth2'] + [c for c in df_ml.columns if any(p in c for p in ['Grade_', 'District_', 'SchYear_'])] #, 'District_', 'SchYear_'      , 'District_', 'Date_'
    elif t_var=='NormWinsr3h2':
        W_cols=['IMR','NormWinsr1h2','NormWinsFluctMonth2'] + [c for c in df_ml.columns if any(p in c for p in ['Grade_', 'District_', 'SchYear_'])] #, 'District_', 'SchYear_'      , 'District_', 'Date_'
    else:
        W_cols=['IMR','NormWinsr1h2', 'NormWinsr3h2'] + [c for c in df_ml.columns if any(p in c for p in ['Grade_', 'District_', 'SchYear_'])] #, 'District_', 'SchYear_'    , 'District_', 'Date_'


    # A. 构建并拟合 LinearDML
    seed = 120      #$$$$
    est = LinearDML(
        model_y=LassoCV(random_state=seed),##@@@
        model_t=LassoCV(random_state=seed),
        # model_y=RandomForestRegressor(n_estimators=100, random_state=seed),
        # model_t=RandomForestRegressor(n_estimators=100, random_state=seed),
        # model_y=GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=seed),
        # model_t=GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=seed),
        # model_y=RidgeCV(), 
        # model_t=RidgeCV(),
        discrete_treatment=False,
        cv=3,
        random_state=seed
    )
    

    # # 创建一个组合标签，例如 "1-0-1" 代表 "私立-农村-男生"
    # combined_str = df_ml[X_cols].astype(str).agg('-'.join, axis=1)
    # # 自动计算平衡权重：样本越少的组合，权重越高
    # # compute_sample_weight 会确保所有类别的权重总和相等
    # sw = compute_sample_weight(class_weight='balanced', y=combined_str)

    # 拟合
    # 首先将Y和T对W与X回归，得到残差；然后T的残差对Y的残差回归，系数即为调节效应tao(x)；然后再用X各个特征回归这个tao(x)；
    # 得到的X的各个特征的系数即为其各个特征对“T影响Y”的贡献，亦即T:X交互项；
    
    # weights = np.where(df_ml['SchLocation'] == 1, 1/0.35, 1/0.65)
    est.fit(df_ml[Y_col], df_ml[t_var], X=df_ml[X_cols], W=df_ml[W_cols]) #, sample_weight=sw
    

    # B. 计算个体处理效应 (HTE)
    hte_values = est.effect(df_ml[X_cols])
    df_ml[f'HTE_{t_var}'] = hte_values
    
    # C. 全模型统计摘要 (这是 LinearDML 的核心优势，直接给出系数 P 值)
    print("\n--- 1. LinearDML 阶段二系数汇总 (直接对应交互项系数) ---")
    print(est.summary()) # 这一步直接产出类似回归表的统计结果

    # D. 显著效应样本占比
    inf_full = est.effect_inference(df_ml[X_cols])
    df_ml[f'p_value_{t_var}'] = inf_full.pvalue()
    print(f"\n--- 2. 个体层面显著效应样本占比 (p < 0.05): {(df_ml[f'p_value_{t_var}'] < 0.05).mean():.2%}")

    # E. 各变量组合的具体效应预测
    print("\n--- 3. 各变量组合的具体效应预测 ---")
    combinations = list(itertools.product([0, 1], repeat=len(X_cols)))
    df_comb = pd.DataFrame(combinations, columns=X_cols)
    inf_comb = est.effect_inference(df_comb)
    df_comb['treatment_effect'] = est.effect(df_comb)
    df_comb['p_value'] = inf_comb.pvalue()
    
    def describe_case(row):
        st = "私立" if row['SchType'] == 1 else "公立"
        sl = "城市" if row['SchLocation'] == 1 else "农村"
        ge = "男生" if row['Gender'] == 1 else "女生"
        return f"{st}-{sl}-{ge}" #f"{st}-{sl}" #f"{st}-{sl}-{ge}"
    
    
    df_comb['Group_Desc'] = df_comb.apply(describe_case, axis=1)
    print(df_comb[['Group_Desc', 'treatment_effect', 'p_value']])

    # F. 分组汇总表生成
    sub_tables = []
    for x_col in X_cols:
        grouped = df_ml.groupby(x_col)[f'HTE_{t_var}'].agg(['mean', 'std', 'count']).reset_index()
        grouped.columns = ['Category_Value', 'Mean_Effect', 'Std_Dev', 'Sample_Size']
        grouped['Moderator'], grouped['Treatment'] = x_col, t_var
        sub_tables.append(grouped)
    final_summary_tables.append(pd.concat(sub_tables, ignore_index=True))


# 结果保存
final_report = pd.concat(final_summary_tables, ignore_index=True)
output_path = r'C:\Users\ASUS\Desktop\Master Thesis\Data of My Thesis\Education Data\important!!\Process Check\Causal_Forest_Final_Results_ExcellenceRate.csv'
final_report.to_csv(output_path, index=False)
print(f"\n全部分析完成，导出路径: {output_path}")


==================== 正在进行 LinearDML 分析: NormWinsr1h2 ====================

--- 1. LinearDML 阶段二系数汇总 (直接对应交互项系数) ---
                       Coefficient Results                       
            point_estimate stderr zstat  pvalue ci_lower ci_upper
-----------------------------------------------------------------
SchType             -0.036  0.007 -5.036    0.0    -0.05   -0.022
SchLocation         -0.001  0.004 -0.184  0.854   -0.008    0.007
Gender               0.001  0.003  0.203  0.839   -0.006    0.008
                       CATE Intercept Results                      
               point_estimate stderr zstat pvalue ci_lower ci_upper
-------------------------------------------------------------------
cate_intercept          0.003  0.003 1.015   0.31   -0.003    0.008
-------------------------------------------------------------------

<sub>A linear parametric conditional average treatment effect (CATE) model was fitted:
$Y = \Theta(X)\cdot T + g(X, W) + \epsilon$
where for every